Purpose of this notebook:
1) Load the cleaned severity dataset from Notebook 1
2) Create a proxy flag for supplier/upstream-related recalls using keywords
3) Compare severity distribution for supplier-related vs non-supplier recalls
4) Compute key headline metrics (e.g., % severe among supplier-related)
5) Save analysis outputs for Notebook 3 and reporting

Inputs:
data/processed/recalls_severity_clean.csv

Outputs:
data/processed/supplier_severity_table.csv
data/processed/supplier_risk_metrics.csv

In [1]:
import pandas as pd

In [2]:
# -----------------------------
# STEP 0: File paths
# -----------------------------
IN_PATH = "data/processed/recalls_severity_clean.csv"
OUT_TABLE_PATH = "data/processed/supplier_severity_table.csv"
OUT_METRICS_PATH = "data/processed/supplier_risk_metrics.csv"

In [3]:
# -----------------------------
# STEP 1: Load processed data
# -----------------------------
df = pd.read_csv(IN_PATH)

print("Processed dataset loaded")
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))

df.head()

Processed dataset loaded
Shape: (503, 25)

Columns: ['country', 'city', 'address_1', 'reason_for_recall', 'address_2', 'product_quantity', 'code_info', 'center_classification_date', 'distribution_pattern', 'state', 'product_description', 'report_date', 'classification', 'openfda', 'recalling_firm', 'recall_number', 'initial_firm_notification', 'product_type', 'event_id', 'more_code_info', 'recall_initiation_date', 'postal_code', 'voluntary_mandated', 'status', 'severity_score']


,country,city,address_1,reason_for_recall,address_2,product_quantity,code_info,center_classification_date,distribution_pattern,state,...,recall_number,initial_firm_notification,product_type,event_id,more_code_info,recall_initiation_date,postal_code,voluntary_mandated,status,severity_score
0,United States,Davie,4131 SW 47th Ave Ste 1403,Recall initiated as a precautionary measure du...,NaN,"1,990 bottles","UPC No. 632687615989; Lot No. 30661601, Exp. D...",20161025,"FL, MI, MS, and OH.",FL,...,F-0276-2017,Letter,Food,75272,NaN,2016-08-08,33314-4036,Voluntary: Firm initiated,Ongoing,3
1,United States,Miami,13439 NW 19 LANE,Virginia State (VDACS) found Listeria monocyto...,NaN,144 pieces,UPC 635349 000390 Best By dates: 07/01/14 thr...,20141202,"FL, GA. NC, and TN",FL,...,F-0609-2015,"Two or more of the following: Email, Fax, Lett...",Food,69516,20170328.0,2014-10-10,33182,Voluntary: Firm initiated,Terminated,5
2,United States,Seattle,3429 Airport Way S,Coffee Toffee is recalled because pecan is lis...,NaN,24 packages,no codes,20180614,distributed in WA,WA,...,F-1578-2018,Visit,Food,80233,20180625.0,2018-05-25,98134-2139,Voluntary: Firm initiated,Terminated,1
3,United States,Brooklyn,47 Bridgewater St # 57,"Product contains dried peaches, but front labe...",NaN,unknown,UPC CODE: 6868978724496 BEST BEFORE: 11/15/2021,20200424,Unknown,NY,...,F-0921-2020,"Two or more of the following: Email, Fax, Lett...",Food,85364,20210318.0,2020-04-01,11222-3820,Voluntary: Firm initiated,Terminated,1
4,United States,Tipp City,320 N 2nd St,The firm stated that the product contains unde...,NaN,480/20 ib cases,"Product #29973B Code Dates: 10/20/2016, 11/8/...",20170605,Product was sent to one manufacturer in MI,OH,...,F-2326-2017,Letter,Food,77213,20180213.0,2017-05-05,45371,Voluntary: Firm initiated,Terminated,3


In [4]:
# -----------------------------
# STEP 2: Validate required columns
# -----------------------------
required_cols = ["classification", "severity_score", "reason_for_recall"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}. Check Notebook 1 output.")

print("Required columns present:", required_cols)

Required columns present: ['classification', 'severity_score', 'reason_for_recall']


In [5]:
# -----------------------------
# STEP 3: Supplier/upstream keyword proxy
# -----------------------------
# IMPORTANT:
# This is a proxy model, not ground truth.
# It helps approximate supplier/upstream-linked issues when internal data is unavailable.
keywords = [
    "supplier",
    "ingredient",
    "raw material",
    "contamination",
    "listeria",
    "salmonella",
    "allergen",
    "foreign material",
    "undeclared",
    "packaging",
    "label",
    "mislabel",
    "metal",
    "plastic",
    "glass"
]

text = df["reason_for_recall"].fillna("").str.lower()

df["supplier_related_flag"] = text.apply(lambda x: any(k in x for k in keywords))

print("--- Supplier-related flag distribution ---")
flag_counts = df["supplier_related_flag"].value_counts()
print(flag_counts)

--- Supplier-related flag distribution ---
supplier_related_flag
True     401
False    102
Name: count, dtype: int64


In [6]:
# -----------------------------
# STEP 4: Severity mix: supplier vs non-supplier
# -----------------------------
supplier_severity = (
    df.groupby(["supplier_related_flag", "classification"])
      .size()
      .unstack(fill_value=0)
)

print("--- Severity table (supplier vs non-supplier) ---")
supplier_severity

--- Severity table (supplier vs non-supplier) ---


classification,Class I,Class II,Class III
supplier_related_flag,,,
False,9,82,11
True,199,186,16


In [7]:
# -----------------------------
# STEP 5: Key headline metrics (for report / interviews)
# -----------------------------
supplier_only = df[df["supplier_related_flag"] == True].copy()
non_supplier_only = df[df["supplier_related_flag"] == False].copy()

total_supplier = len(supplier_only)
total_non_supplier = len(non_supplier_only)

severe_classes = ["Class I", "Class II"]

severe_supplier = supplier_only["classification"].isin(severe_classes).sum()
severe_non_supplier = non_supplier_only["classification"].isin(severe_classes).sum()

severe_supplier_pct = round((severe_supplier / total_supplier) * 100, 2) if total_supplier else 0
severe_non_supplier_pct = round((severe_non_supplier / total_non_supplier) * 100, 2) if total_non_supplier else 0

# Add one more useful metric: share of all recalls that are supplier-related
supplier_share_pct = round((total_supplier / len(df)) * 100, 2)

metrics = pd.DataFrame([{
    "total_records": len(df),
    "supplier_related_count": total_supplier,
    "supplier_related_share_pct": supplier_share_pct,
    "severe_supplier_count_class_I_II": int(severe_supplier),
    "severe_supplier_pct_class_I_II": severe_supplier_pct,
    "non_supplier_count": total_non_supplier,
    "severe_non_supplier_count_class_I_II": int(severe_non_supplier),
    "severe_non_supplier_pct_class_I_II": severe_non_supplier_pct
}])

print("--- Key metrics ---")
metrics

--- Key metrics ---


,total_records,supplier_related_count,supplier_related_share_pct,severe_supplier_count_class_I_II,severe_supplier_pct_class_I_II,non_supplier_count,severe_non_supplier_count_class_I_II,severe_non_supplier_pct_class_I_II
0,503,401,79.72,385,96.01,102,91,89.22


In [8]:
# -----------------------------
# STEP 6: Save outputs for Notebook 3 / reporting
# -----------------------------
supplier_severity.to_csv(OUT_TABLE_PATH)
metrics.to_csv(OUT_METRICS_PATH, index=False)

print(f"Saved severity table to: {OUT_TABLE_PATH}")
print(f"Saved metrics to: {OUT_METRICS_PATH}")


Saved severity table to: data/processed/supplier_severity_table.csv
Saved metrics to: data/processed/supplier_risk_metrics.csv
